<a href="https://colab.research.google.com/github/fdx-hw/cosc-650/blob/main/week5_rag.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Week 5 (starter): RAG Pipeline with Retrieval Evaluation

Retrieval runs fully local and free; only generation needs a key. This notebook uses the Anthropic API instead. Set `ANTHROPIC_API_KEY`, else generation is skipped with a
notice. Dependencies: `sentence-transformers`,
`faiss-cpu`, `anthropic`.

In [1]:
%pip install -q sentence-transformers faiss-cpu anthropic

In [2]:
import os, re, pathlib, numpy as np
os.environ['HF_HOME'] = str((pathlib.Path('.') / '.hf_cache').resolve())
from sentence_transformers import SentenceTransformer
import faiss
embedder = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')

DOCS = {
    'networking': '''## Networking
The Orbit server listens on TCP port 7700 by default. Gossip runs on 7711.
Clients should connect to port 7700; port 7711 is for inter-node gossip only
and should not be exposed publicly.''',

    'storage': '''## Storage
Orbit persists data with an LSM-tree storage engine.
Writes go to an in-memory memtable first and are flushed to disk as
immutable SSTable segments once the memtable fills.''',

    'replication': '''## Replication
Each key is replicated with a default replication factor of 3.
Default consistency is quorum, meaning a write or read must be
acknowledged by at least two of the three replicas to succeed.''',

    'limits': '''## Limits
The maximum value size is 16 MB. A batch may contain at most 1000 operations.
Keys are limited to 1 KB. Requests exceeding any of these limits are
rejected with a client-side validation error before reaching storage.''',

    'security': '''## Security
Clients authenticate with API tokens. Traffic is encrypted with TLS.
Tokens are scoped per-namespace and can be revoked instantly through
the admin API without restarting the server.''',

    'backup': '''## Backup
Orbit takes incremental snapshots every 6 hours by default, written to
the configured backup directory. A full snapshot can be triggered manually
with orbitctl snapshot --full. Snapshots older than 7 days are pruned
automatically unless retention is overridden in orbit.yaml.''',

    'monitoring': '''## Monitoring
Orbit exposes Prometheus-compatible metrics on port 7720 at /metrics.
Health checks are available at /healthz on the same port. The metrics
port is separate from the client port and is not encrypted by default.''',

    'deployment': '''## Deployment
Orbit is configured through a single YAML file, orbit.yaml, loaded at
startup. Environment variables prefixed with ORBIT_ override values from
the file. The default data directory is /var/lib/orbit, and the default
config search path is /etc/orbit/orbit.yaml.''',
}
print('doc count:', len(DOCS), '| total chars:', sum(len(t) for t in DOCS.values()))

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

doc count: 8 | total chars: 1789


In [3]:
def chunk_fixed(text, size, overlap):
    text = re.sub(r'\s+', ' ', text).strip(); out, i = [], 0
    while i < len(text):
        out.append(text[i:i+size]); i += size - overlap
    return out
def chunk_paragraph(text):
    return [re.sub(r'\s+',' ',p).strip() for p in text.split('\n\n') if p.strip()]

def build_config(chunk_fn, *args):
    """Run a chunker over every document, returning a flat chunk list plus, for each
    chunk, which source doc it came from -- kept for Part 2's relevance labels and for
    tracing exactly where a fact came from in Part 4."""
    chunks, sources = [], []
    for name, text in DOCS.items():
        pieces = chunk_fn(text, *args) if args else chunk_fn(text)
        for c in pieces:
            chunks.append(c); sources.append(name)
    return chunks, sources

configs = {
    'small (120/20)': build_config(chunk_fixed, 120, 20),
    'large (320/40)': build_config(chunk_fixed, 320, 40),
    'by-paragraph':   build_config(chunk_paragraph),
}
for n, (c, s) in configs.items():
    print(f'{n:16s} -> {len(c)} chunks')

def build(chunks):
    e = embedder.encode(chunks, normalize_embeddings=True).astype('float32')
    idx = faiss.IndexFlatIP(e.shape[1]); idx.add(e); return idx
def retrieve(idx, chunks, q, k=3):
    qe = embedder.encode([q], normalize_embeddings=True).astype('float32')
    ids = idx.search(qe, k)[1][0]
    return [chunks[i] for i in ids if i >= 0], [int(i) for i in ids if i >= 0]

indices = {n: build(c) for n, (c, s) in configs.items()}

small (120/20)   -> 22 chunks
large (320/40)   -> 9 chunks
by-paragraph     -> 8 chunks


## Part 2: Evaluate retrieval (compare chunking configs)
A query is answered only if the retrieved context contains the answer fact. This is the model-free half of answer quality.

In [4]:
QA = [
    ('what port does orbit listen on for clients', '7700', 'networking'),
    ('what port is used for gossip between nodes', '7711', 'networking'),
    ('what storage engine does orbit use', 'LSM-tree', 'storage'),
    ('what happens when the memtable fills up', 'flushed to disk', 'storage'),
    ('what is the default replication factor', 'replication factor of 3', 'replication'),
    ('what consistency level is used by default', 'quorum', 'replication'),
    ('what is the maximum value size', '16 MB', 'limits'),
    ('how many operations can a batch contain', '1000 operations', 'limits'),
    ('how do clients authenticate', 'API tokens', 'security'),
    ('how often are snapshots taken', 'every 6 hours', 'backup'),
    ('what port serves metrics', '7720', 'monitoring'),
    ('where can i check server health', '/healthz', 'monitoring'),
    ('how is orbit configured', 'configured through a single YAML file', 'deployment'),
    ('what environment variable prefix overrides config', 'ORBIT_', 'deployment'),
]
print(f'{len(QA)} queries defined (>= 10 required)')

def chunk_relevance(chunks, fact):
    """Ground truth, decided in advance: a chunk counts as relevant to a query iff it
    contains that query's pre-written answer fact. The fact strings above were written
    down before any chunking config was picked, so the *criterion* is fixed in advance,
    even though the resulting index set is necessarily computed per config (chunk
    boundaries differ between configs).

    Caveat: an exact-substring rule can mark a chunk relevant when it contains the fact
    string without the context that actually answers the question. This wasn't just
    theoretical here -- the original fact for 'how is orbit configured' was the bare
    string 'orbit.yaml', which also appears in the backup doc's unrelated sentence about
    snapshot-retention overrides, wrongly counting that chunk as relevant too. Fixed by
    picking a longer, more specific fact string that only appears in the intended doc;
    every other fact in QA was checked the same way and is unique to its source doc."""
    return {i for i, c in enumerate(chunks) if fact.lower() in c.lower()}

def precision_recall_at_k(config_name, k=3, verbose=True):
    chunks, sources = configs[config_name]
    idx = indices[config_name]
    p_sum, r_sum, n = 0.0, 0.0, 0
    for q, fact, doc in QA:
        _, retrieved_ids = retrieve(idx, chunks, q, k=k)
        retrieved = set(retrieved_ids)
        relevant = chunk_relevance(chunks, fact)
        tp = len(retrieved & relevant)
        precision = tp / len(retrieved) if retrieved else 0.0
        recall = tp / len(relevant) if relevant else 0.0
        p_sum += precision; r_sum += recall; n += 1
        if verbose:
            note = '  <-- fact matched 0 intact chunks (chunking failure)' if not relevant else ''
            print(f'  {q!r}: retrieved={sorted(retrieved)} relevant={sorted(relevant)} '
                  f'tp={tp} precision={precision:.2f} recall={recall:.2f}{note}')
    return p_sum / n, r_sum / n

for name in configs:
    print(f'=== {name} ===')
    p, r = precision_recall_at_k(name, k=3)
    print(f'{name:16s} precision@3={p:.2f}  recall@3={r:.2f}')
    print()

14 queries defined (>= 10 required)
=== small (120/20) ===
  'what port does orbit listen on for clients': retrieved=[0, 20, 21] relevant=[0, 1] tp=1 precision=0.33 recall=0.50
  'what port is used for gossip between nodes': retrieved=[0, 1, 17] relevant=[0, 1] tp=2 precision=0.67 recall=1.00
  'what storage engine does orbit use': retrieved=[3, 20, 21] relevant=[3] tp=1 precision=0.33 recall=1.00
  'what happens when the memtable fills up': retrieved=[3, 4, 18] relevant=[4] tp=1 precision=0.33 recall=1.00
  'what is the default replication factor': retrieved=[5, 6, 18] relevant=[5] tp=1 precision=0.33 recall=1.00
  'what consistency level is used by default': retrieved=[5, 6, 9] relevant=[5, 6] tp=2 precision=0.67 recall=1.00
  'what is the maximum value size': retrieved=[8, 9, 10] relevant=[8] tp=1 precision=0.33 recall=1.00
  'how many operations can a batch contain': retrieved=[5, 8, 13] relevant=[8] tp=1 precision=0.33 recall=1.00
  'how do clients authenticate': retrieved=[11, 17

**Results.**

```
small (120/20)   precision@3=0.38  recall@3=0.96
large (320/40)   precision@3=0.33  recall@3=1.00
by-paragraph     precision@3=0.33  recall@3=1.00
```

`large (320/40)` and `by-paragraph` tie exactly again, for the same reason as before: every
doc here is short enough that both configs land at one chunk per doc, so every one of the 14
queries gets exactly 1 relevant chunk and exactly 1 true positive, recall@3=1.00 across the
board, and precision@3 stuck at 1/3, since only one of the three retrieved slots can ever be
right when there's only one correct chunk to find. `large` still reports 9 chunks instead of
8 (the same trailing-junk-chunk bug in its own chunker, producing a near-empty `'yaml.'` chunk
from the `backup` doc), and the per-query breakdown confirms that junk chunk is never once
retrieved across all 14 queries, so it doesn't move the numbers either.

`small (120/20)` moves in both directions, and now that the per-query IDs are visible, the
real cause is more specific than a general "chunking hurts recall" story. Precision rises
above the 1/3 ceiling on 2 of the 14 queries (0.67 each), both happen when a fact-bearing
phrase occurs twice in a doc (e.g. "port 7700" is stated twice in the networking doc), so two
different chunks both legitimately count as relevant and both land in the same top-3.
Recall drops below 1.00 on exactly one query out of 14: "what port does orbit listen on for
clients" retrieves only one of the two chunks containing '7700' and picks up two unrelated
deployment chunks instead. A genuine near-miss, not the `/healthz` sentence split from Part
4. That split query actually keeps recall@3=1.00 in every config, small included, since the
trailing chunk alone still contains the intact `/healthz` string; its failure shows up in
generation (Part 4), not in this recall number. Worth being explicit about that correction:
before the per-query printout existed, it looked reasonable to assume the recall dip was the
Part 4 split catching up with the metric, but the real breakdown shows a different, one-off
retrieval miss is entirely responsible. Net: precision@3=0.38 (above the 0.33 ceiling),
recall@3=0.96 (just under the other two configs' perfect 1.00) which is still a real tradeoff
between keeping facts intact and keeping retrieval tightly on-topic, just not for the reason
originally assumed.

One limitation worth naming directly: relevance here is decided by exact fact-substring
matching, not manual or semantic judgment. That's reproducible and fixed in advance, but it
can miscount a chunk as relevant if it happens to contain the fact string out of context,
which is exactly what happened with an earlier version of one fact here (`'orbit.yaml'`
matched an unrelated sentence in the `backup` doc) before it was caught and replaced with a
more specific phrase. The opposite failure is also possible with this method: a chunk that's
genuinely about the right topic but phrases the fact differently would be missed entirely.
Manual or semantic labeling avoids both problems but doesn't scale past a handful of queries
and trades away reproducibility for judgment calls. There's no single correct choice here,
just a tradeoff worth stating rather than leaving implicit.

## Part 3: Generate (needs a key)
Build a grounded prompt from the retrieved chunks and answer. Uses the Anthropic API
(Claude) instead of the assignment's stated Gemini free tier.

In [5]:
def _get_api_key():
    try:
        from google.colab import userdata
        key = userdata.get('ANTHROPIC_API_KEY')
        if key:
            return key
    except Exception:
        pass
    key = os.environ.get('ANTHROPIC_API_KEY', '').strip()
    if key:
        return key
    try:
        import getpass
        return getpass.getpass('ANTHROPIC_API_KEY (leave blank to skip generation): ').strip()
    except Exception:
        return ''

_API_KEY = _get_api_key()
if _API_KEY:
    import anthropic
    _client = anthropic.Anthropic(api_key=_API_KEY)
GEN_MODEL = 'claude-haiku-4-5-20251001'

def claude_chat(messages, model=GEN_MODEL, **kw):
    if not _API_KEY:
        return None
    return _client.messages.create(model=model, max_tokens=512, messages=messages, **kw).content[0].text

def answer(q, config='by-paragraph', k=3):
    chunks, sources = configs[config]
    ctx, ids = retrieve(indices[config], chunks, q, k=k)
    prompt = 'Answer only from the context.\nContext:\n- ' + '\n- '.join(ctx) + f'\nQuestion: {q}\nAnswer:'
    out = claude_chat([{'role':'user','content':prompt}])
    return (out if out is not None else '[API-BLOCKED] set ANTHROPIC_API_KEY to generate; grounded prompt was built'), ctx

resp, ctx = answer('what port does orbit use')
print(resp)

Based on the context, Orbit uses the following ports:

1. **Port 7700** - TCP port for client connections (default)
2. **Port 7711** - Gossip port for inter-node communication (should not be exposed publicly)
3. **Port 7720** - Metrics port for Prometheus-compatible metrics and health checks


## Part 4: a real failure -- a chunking choice splits a fact from the words that would find it

**First, a plain check with no model involved.** I compared every sentence in the corpus
against every chunk in every config, just string matching, no embeddings needed. Under
`small (120/20)`, several sentences get cut in half. Here's one: the monitoring doc's line
"Health checks are available at /healthz on the same port" lands right on a chunk boundary:

```
chunk A: '...Prometheus-compatible metrics on port 7720 at /metrics. Health checks are available at /heal'
chunk B: 'e available at /healthz on the same port. The metrics port is separate from...'
```

Neither chunk has the whole phrase. Chunk A cuts off right before "thz", chunk B picks up
mid-word and never gets the "Health checks ar[e]" part at the start. `large (320/40)` and
`by-paragraph` don't have this problem -- every doc here is short enough that both of them
keep each sentence whole, which I checked the same way.

**Then I ran it live, and the effect was worse than I expected.** I asked "where can i check
server health" against `small (120/20)`:

```
retrieved chunks (small 120/20):
 - '## Monitoring Orbit exposes Prometheus-compatible metrics on port 7720 at /metrics. Health checks are available at /heal'
 - 'e available at /healthz on the same port. The metrics port is separate from the client port and is not encrypted by defa'
 - 'uld connect to port 7700; port 7711 is for inter-node gossip only and should not be exposed publicly.'

answer: Based on the context, you can check server health at:
1. `/healthz` on port 7720
2. `/heal` on port 7720
Both endpoints are available on the metrics port 7720.
```

Both halves of the split sentence actually made it into the top-3, so retrieval did its job
here, the right information wasn't missing. The problem showed up in generation instead:
Claude read the truncated `/heal` in chunk A and treated it as a second, real endpoint,
answering with two health-check paths when there's only one. I think the "answer only from
context" instruction made this worse rather than better. It pushes the model to pull
literally from whatever text is in front of it instead of asking whether `/heal` even looks
like a complete path. Worth noting too: the third retrieved chunk is an unrelated networking
snippet about client/gossip ports. It didn't end up in the answer, but it's the same
precision problem already showing up in Part 2's numbers -- small chunks pull in more
off-topic material.

**One more honest note: this isn't 100% reproducible.** Running that same cell again with the
identical retrieved context, Claude sometimes answers correctly instead of just "`/healthz`
on port 7720," no second endpoint. The API call isn't pinned to `temperature=0`, so the same
truncated `/heal` fragment doesn't hallucinate into a fake endpoint on every single run, only
some of them. That doesn't make this any less of a real failure. The retrieved context is
identical either way, and a chunking choice that leaves the model's answer up to chance on
otherwise-identical input is arguably a worse failure than a deterministic one, since it's the
kind of bug that can pass a quick manual check and still fail in production. It does mean the
mitigation (bigger chunks or paragraph/sentence boundaries) is fixing the actual cause, the
truncated text, rather than papering over a one-off fluke.

In [8]:
q = 'where can i check server health'
ctx, ids = retrieve(indices['small (120/20)'], configs['small (120/20)'][0], q, k=3)
print('retrieved chunks (small 120/20):')
for c in ctx:
    print(' -', repr(c))
resp, _ = answer(q, config='small (120/20)')
print()
print('answer:', resp)

retrieved chunks (small 120/20):
 - '## Monitoring Orbit exposes Prometheus-compatible metrics on port 7720 at /metrics. Health checks are available at /heal'
 - 'e available at /healthz on the same port. The metrics port is separate from the client port and is not encrypted by defa'
 - 'uld connect to port 7700; port 7711 is for inter-node gossip only and should not be exposed publicly.'

answer: Based on the context, you can check server health at:

1. `/healthz` - available on port 7720
2. `/heal` - available on port 7720

Both health check endpoints are available on port 7720 (the same port where Prometheus-compatible metrics are exposed).


**Mitigation.** Increase chunk size (or overlap) so sentences aren't cut mid-clause, or
chunk by sentence/paragraph boundary instead of a fixed character count, both `large
(320/40)` and `by-paragraph` already avoid this for the current corpus, at the cost of
bundling more unrelated content into each chunk (see the Part 2 precision/recall tradeoff).

## Part 5: Submit
Open a pull request with a result summary and review a classmate's Week 5 PR (the term's
formal peer review). Use the rubric attached to the Canvas assignment for grading criteria
and point values.